# <font color='gold'>LLM ABSA annotation</font>


In [1]:
import os
import pandas as pd
from sampling_util import data_processing

#===========================================================
batch_size = 100  # this is per_reveiw_type number
random_state = 73 # sampling random seed
#===========================================================

input_file = '../../data/2_sampled/combined_data_6.5k.csv'
input_file_with_batchID = input_file.replace('data/2_sampled/', 'data/3_batched/')
batch_dict = 'batched_data_dict.py'

print(f'\nInput:    {input_file}\nOutput:   {input_file_with_batchID}\n\nbatch_dict:  \033[35m{batch_dict}\033[0m\n')



Input:    ../../data/2_sampled/combined_data_6.5k.csv
Output:   ../../data/3_batched/combined_data_6.5k.csv

batch_dict:  batched_data_dict.py



## <font color='cornflowerblue'>1) Read combined samples that has course, teacher, university reviews</font>


In [2]:
input_df = pd.read_csv(input_file)
print(input_df.shape, list(input_df.columns.values), '\n', sep='\n')
input_df

(6500, 10)
['type', 'source', 'id', 'rating', 'num_token', 'sample_criteria', 'review_text', 'rating_bin', 'rating_cat', 'length_bin']




,type,source,id,rating,num_token,sample_criteria,review_text,rating_bin,rating_cat,length_bin
0,teacher review,data1_19145,1_45_2639_18911,5.0,74,"Pos_(61.0, 81.0]",Dr. Zhong is a really good chemistry teacher. ...,"(4.0, 5.0]",Pos,"(61.0, 81.0]"
1,teacher review,data1_19145,1_206_452_3440,1.5,56,"Neg_(41.0, 61.0]",I have her for behav. neuro and she is terribl...,"(0.999, 2.5]",Neg,"(41.0, 61.0]"
2,teacher review,data1_19145,1_93_2121_1031,1.0,77,"Neg_(61.0, 81.0]",the prof. is real crap. it is a shame that the...,"(0.999, 2.5]",Neg,"(61.0, 81.0]"
3,teacher review,data1_19145,1_89_50_4831,3.5,42,"Conf/Neu_(41.0, 61.0]",He is a great guy and is pretty funny in class...,"(2.5, 4.0]",Conf/Neu,"(41.0, 61.0]"
4,teacher review,data1_19145,1_88_396_5468,1.0,44,"Neg_(41.0, 61.0]","Plain and simple, this guy is HORRIBLE! He not...","(0.999, 2.5]",Neg,"(41.0, 61.0]"
...,...,...,...,...,...,...,...,...,...,...
6495,university review,data3_557,3_343_393_290,4.8,15,"Pos_(0.999, 21.0]","University is absolutely as good as expected, ...","(4.0, 5.0]",Pos,"(0.999, 21.0]"
6496,university review,data3_557,3_118_412_64,5.0,25,"Pos_(21.0, 41.0]","Brilliant University, perfect fit for me even ...","(4.0, 5.0]",Pos,"(21.0, 41.0]"
6497,university review,data3_557,3_334_178_52,4.2,12,"Pos_(0.999, 21.0]",Great uni! Loads of food outlets and study are...,"(4.0, 5.0]",Pos,"(0.999, 21.0]"
6498,university review,data3_557,3_427_169_213,4.6,16,"Pos_(0.999, 21.0]","The campus is amazing, people are all fantasti...","(4.0, 5.0]",Pos,"(0.999, 21.0]"


## <font color='cornflowerblue'>2) Replace \\'  with '<font>


In [3]:
ids_to_replace = set(input_df.loc[input_df.review_text.str.contains("\\'")].id.unique())

xx = input_df.copy()
input_df.review_text = input_df.review_text.str.replace("\\'","'")
input_df.review_text = input_df.review_text.apply(lambda x: data_processing.add_space_around_emoji(x))

display(xx.loc[xx.id.isin(ids_to_replace)].sort_values(by='id'))

display(input_df.loc[input_df.id.isin(ids_to_replace)].sort_values(by='id').head(5))

print(input_df.loc[input_df.id=='3_545_1_195'].review_text.values)

,type,source,id,rating,num_token,sample_criteria,review_text,rating_bin,rating_cat,length_bin
2630,teacher review,data1_19145,1_101_2523_18606,4.5,24,"Pos_(21.0, 41.0]",I don\'t understand the lukewarm reviews here....,"(4.0, 5.0]",Pos,"(21.0, 41.0]"
680,teacher review,data1_19145,1_101_2596_9981,5.0,58,"Pos_(41.0, 61.0]",I learned and maintained more information from...,"(4.0, 5.0]",Pos,"(41.0, 61.0]"
1950,teacher review,data1_19145,1_104_1870_16799,2.5,68,"Neg_(61.0, 81.0]",She\'s a very nice lady that makes jokes and s...,"(0.999, 2.5]",Neg,"(61.0, 81.0]"
2119,teacher review,data1_19145,1_104_1870_5497,1.0,41,"Neg_(21.0, 41.0]",Worst prof ever!! Has no clue what she\'s talk...,"(0.999, 2.5]",Neg,"(21.0, 41.0]"
2036,teacher review,data1_19145,1_104_189_13817,4.0,31,"Conf/Neu_(21.0, 41.0]",Very knowledgeable and helpful. Tests are hard...,"(2.5, 4.0]",Conf/Neu,"(21.0, 41.0]"
...,...,...,...,...,...,...,...,...,...,...
6071,university review,data3_557,3_545_1_195,4.7,53,"Pos_(41.0, 61.0]","Exeter is always welcoming, efficient student/...","(4.0, 5.0]",Pos,"(41.0, 61.0]"
6087,university review,data3_557,3_552_1_182,4.7,18,"Pos_(0.999, 21.0]",Exeter is a great uni! Everything you need is ...,"(4.0, 5.0]",Pos,"(0.999, 21.0]"
6467,university review,data3_557,3_553_1_182,4.7,29,"Pos_(21.0, 41.0]",Brilliant university; beautiful campus; active...,"(4.0, 5.0]",Pos,"(21.0, 41.0]"
6385,university review,data3_557,3_556_1_182,4.7,15,"Pos_(0.999, 21.0]",ITS AMAZING! I love my university and wouldn't...,"(4.0, 5.0]",Pos,"(0.999, 21.0]"


,type,source,id,rating,num_token,sample_criteria,review_text,rating_bin,rating_cat,length_bin
2630,teacher review,data1_19145,1_101_2523_18606,4.5,24,"Pos_(21.0, 41.0]",I don't understand the lukewarm reviews here. ...,"(4.0, 5.0]",Pos,"(21.0, 41.0]"
680,teacher review,data1_19145,1_101_2596_9981,5.0,58,"Pos_(41.0, 61.0]",I learned and maintained more information from...,"(4.0, 5.0]",Pos,"(41.0, 61.0]"
1950,teacher review,data1_19145,1_104_1870_16799,2.5,68,"Neg_(61.0, 81.0]",She's a very nice lady that makes jokes and sh...,"(0.999, 2.5]",Neg,"(61.0, 81.0]"
2119,teacher review,data1_19145,1_104_1870_5497,1.0,41,"Neg_(21.0, 41.0]",Worst prof ever!! Has no clue what she's talki...,"(0.999, 2.5]",Neg,"(21.0, 41.0]"
2036,teacher review,data1_19145,1_104_189_13817,4.0,31,"Conf/Neu_(21.0, 41.0]",Very knowledgeable and helpful. Tests are hard...,"(2.5, 4.0]",Conf/Neu,"(21.0, 41.0]"


["Exeter is always welcoming, efficient student/staff liaison, the communication between departments and students for exams, results and module changes/problems is brilliant. The library is efficient and 'well-stocked', and the campus is all around beautiful. I feel like I'm in a bubble of paradise here."]


## <font color='cornflowerblue'>3) Select text_review content, make a map with entry ids</font>


In [4]:
# shuffle rows before making batches
input_df = input_df.sample(frac=1, random_state=random_state).reset_index(drop=True)
input_df

,type,source,id,rating,num_token,sample_criteria,review_text,rating_bin,rating_cat,length_bin
0,teacher review,data1_19145,1_65_2377_7868,1.5,41,"Neg_(21.0, 41.0]",She will only accept .docx forms in microsoft ...,"(0.999, 2.5]",Neg,"(21.0, 41.0]"
1,teacher review,data1_19145,1_82_24_8586,4.5,46,"Pos_(41.0, 61.0]",Great Teacher! Learned a lot! If coming to cla...,"(4.0, 5.0]",Pos,"(41.0, 61.0]"
2,teacher review,data1_19145,1_40_1006_19051,5.0,8,"Pos_(0.999, 21.0]",Not so easy class; great instructor!,"(4.0, 5.0]",Pos,"(0.999, 21.0]"
3,course review,data2_14810,2_4496_234,0.0,51,"Neg_(41.0, 61.0]",Studied for the exam literally the day of...en...,[0.0],Neg,"(41.0, 61.0]"
4,university review,data3_557,3_188_353_230,4.6,34,"Pos_(21.0, 41.0]","The campus is very green, and a beautiful plac...","(4.0, 5.0]",Pos,"(21.0, 41.0]"
...,...,...,...,...,...,...,...,...,...,...
6495,course review,data2_14810,2_4559_1341,1.0,54,"Pos_(41.0, 61.0]",I have a ridiculously hard time with writing p...,[1.0],Pos,"(41.0, 61.0]"
6496,teacher review,data1_19145,1_149_1868_802,1.0,28,"Neg_(21.0, 41.0]",He should be a middle school teacher not a col...,"(0.999, 2.5]",Neg,"(21.0, 41.0]"
6497,course review,data2_14810,2_4098_624,1.0,21,"Pos_(0.999, 21.0]","Simple, straightforward lab that pulls a lot f...",[1.0],Pos,"(0.999, 21.0]"
6498,teacher review,data1_19145,1_187_210_13715,5.0,61,"Pos_(41.0, 61.0]",Where do I begin! it was a tough class and req...,"(4.0, 5.0]",Pos,"(41.0, 61.0]"


In [5]:
input_df.insert(loc=0, column="batch_id", value= input_df.index // batch_size + 1)
input_df = input_df.reset_index()
input_df

,index,batch_id,type,source,id,rating,num_token,sample_criteria,review_text,rating_bin,rating_cat,length_bin
0,0,1,teacher review,data1_19145,1_65_2377_7868,1.5,41,"Neg_(21.0, 41.0]",She will only accept .docx forms in microsoft ...,"(0.999, 2.5]",Neg,"(21.0, 41.0]"
1,1,1,teacher review,data1_19145,1_82_24_8586,4.5,46,"Pos_(41.0, 61.0]",Great Teacher! Learned a lot! If coming to cla...,"(4.0, 5.0]",Pos,"(41.0, 61.0]"
2,2,1,teacher review,data1_19145,1_40_1006_19051,5.0,8,"Pos_(0.999, 21.0]",Not so easy class; great instructor!,"(4.0, 5.0]",Pos,"(0.999, 21.0]"
3,3,1,course review,data2_14810,2_4496_234,0.0,51,"Neg_(41.0, 61.0]",Studied for the exam literally the day of...en...,[0.0],Neg,"(41.0, 61.0]"
4,4,1,university review,data3_557,3_188_353_230,4.6,34,"Pos_(21.0, 41.0]","The campus is very green, and a beautiful plac...","(4.0, 5.0]",Pos,"(21.0, 41.0]"
...,...,...,...,...,...,...,...,...,...,...,...,...
6495,6495,65,course review,data2_14810,2_4559_1341,1.0,54,"Pos_(41.0, 61.0]",I have a ridiculously hard time with writing p...,[1.0],Pos,"(41.0, 61.0]"
6496,6496,65,teacher review,data1_19145,1_149_1868_802,1.0,28,"Neg_(21.0, 41.0]",He should be a middle school teacher not a col...,"(0.999, 2.5]",Neg,"(21.0, 41.0]"
6497,6497,65,course review,data2_14810,2_4098_624,1.0,21,"Pos_(0.999, 21.0]","Simple, straightforward lab that pulls a lot f...",[1.0],Pos,"(0.999, 21.0]"
6498,6498,65,teacher review,data1_19145,1_187_210_13715,5.0,61,"Pos_(41.0, 61.0]",Where do I begin! it was a tough class and req...,"(4.0, 5.0]",Pos,"(41.0, 61.0]"


In [6]:
input_df.to_csv(input_file_with_batchID, index=False)

print(f'\ninputdf with batch_id written to \033[32m{input_file_with_batchID}\033[0m~!\n')


inputdf with batch_id written to ../../data/3_batched/combined_data_6.5k.csv~!



In [7]:
if len(input_df.id.unique())==len(input_df):
    print("\033[32mAll good!\n\033[0m") 
    
    input_dict = {batch_number: {} for batch_number in input_df.batch_id.unique()}
    #==================================================
    for batch_number in input_df.batch_id.unique():
        input_dict[batch_number] = input_df.loc[input_df.batch_id == batch_number][['id', 'review_text']].set_index('id').to_dict()['review_text'] 
    print(set(len(subdict) for subdict in input_dict.values()))
    #================================================== 
else:
    display(input_df.loc[input_df.id.duplicated(keep='first')].sort_values(by='id'))
    raise ValueError("ids not unique, check!!!!")
    
input_dict.keys()

All good!

{100}


dict_keys([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65])

In [8]:
original_ids = sorted(input_df.id)
batched_ids = sorted([key for subdict in input_dict.values() for key in subdict.keys()])

if original_ids == batched_ids:
    print("All good!")
    
    if not os.path.exists(batch_dict): 
        with open (batch_dict, 'w', encoding='utf-8') as dictfile: 
            print(batch_dict.replace('.py','='), input_dict, file=dictfile)    
            print(f"\n\033[33mwriting to dict: {batch_dict}\033[0m\n")
    else:
        print(f"\n\u001b[4m\u001b[44m{batch_dict} file already written, check if need overwrite\033[0m\n")

else:
    print([i for i in original_ids if i not in batched_ids], '\n\n')
    raise ValueError("The post-batching subdf ids are not the same as inputdf all id collections, check!!!")

All good!

writing to dict: batched_data_dict.py

